In [153]:
%history -f text.txt

In [ ]:
## USE PROCESS: CATEGORIZATION. CATEGFORIATION: DEFINITION WORD: Metric as an example.
# How to get Optimization and Loss Functions.
# Update ML to have all Parameters?

In [1]:
import pandas as pd
import numpy as np
import datetime

import sys
sys.path.append("/Users/derekdewald/Documents/Python/Github_Repo/d_py_functions")

from objects_automated import object_dict

### Local Files

In [2]:
consolidated_df = pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/consolidated_dataset.xlsx')
knowledge_base_df = pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/knowledge_base.xlsx')

### Google Sheets

In [2]:
link_df = pd.read_csv(object_dict['csv_links']['python_object']['d_links'])
notes_df = pd.read_csv(object_dict['csv_links']['python_object']['google_notes_csv'])
definition_df = pd.read_csv(object_dict['csv_links']['python_object']['google_definition_csv'])
technical_notes = pd.read_csv(object_dict['csv_links']['python_object']['technical_notes'])

In [24]:
def generate_files_for_streamlit(
    definition_df=pd.DataFrame(),
    notes_df=pd.DataFrame(),
    generate_excel_files=True
):

    '''
    Definition:
        Process Utilized to Combine Notes/ Definitions and Logic into Knowledge Base, which is utilized to Create, Processes, Parameters.
    Parameters:
        notes_df (dataframe): Dataframe containing Notes from Google. Default is none and it will pull directly from Google.
        definition_df (dataframe): Dataframe containing Definitions from Google. Default is none and it will pull directly from Google.
        manual_object_df (dataframe): Dataframe containining Dataframe of Parameters, generated from Python Process _____, which converts lists in objects_manual.py.
        export_location(str): Name of File to export excel file to. If Blank, returns nothing
    Returns:
        Excel File
    Date Created:
        02-Jul-26
    Date Last Modified:
        22-Jul-26
    Process:
        Definition
    Categorization:
        Definition
    Usage:
        d = generate_knowledgebase(notes_df,definition_df,manual_object_df)
        d = generate_knowledgebase()
    Notes:
        22-Jul - Overhauled merge. Attempted to streamline, simplify and reduce duplication. Increase Visability.
        28-Jul - Originally Named - generate_knowledge_base, stream lined to remove usage of Dictionary, which were complex and operationally inefficient.
    '''
    
    if len(definition_df)==0:
        definition_df = pd.read_csv(object_dict['csv_links']['python_object']['google_definition_csv'])

    if len(notes_df)==0:
        notes_df = pd.read_csv(object_dict['csv_links']['python_object']['google_notes_csv']).fillna('')

    # Before Merging Files. Update Notes to include Definitions from any item which is a Process from Definition.

    definition_df = definition_df[['Process','Categorization','Word',"Definition"]].fillna('').copy()
    notes_df = notes_df[['Process','Categorization','Word',"Definition"]].fillna('').copy()

    notes_df1 = notes_df.merge(definition_df[definition_df['Word']=='Definition'][['Process','Definition']].rename(columns={'Process':"Word",'Definition':'Definition_'}),on='Word',how='left').fillna("")
    notes_df1['Definition'] = np.where((notes_df1['Categorization']=='Process Step')&(notes_df1['Definition']==""),notes_df1['Definition_'],notes_df1['Definition'])
    notes_df1.drop('Definition_',axis=1,inplace=True)

    # CREATE KNOWLEDGE File, which is simply everything combined Together
    knowledge_base_df = pd.concat([definition_df,notes_df1]).reset_index(drop=True)
    
    # Create 2 Distinct Files Processes. Not Processes
    processes = knowledge_base_df[knowledge_base_df['Categorization']=='Process'].reset_index(drop=True).reset_index().rename(columns={'index':'PROC_ORDER'})
    not_processes = knowledge_base_df[knowledge_base_df['Categorization']!='Process'].reset_index(drop=True).reset_index().rename(columns={'index':'NP_ORDER'})

    #return knowledge_base_df,processes,not_processes

    # Create a Supplemental File which are effectively Items which need to be merged into Processes to move from the Straw Man Process to a more fullsome 
    supplemental = not_processes.merge(knowledge_base_df[['Word','Process']].drop_duplicates().rename(columns={'Process':'Process_','Word':"Process"}),on='Process',how='inner')
    supplemental['Word_'] = supplemental['Word'].copy()
    supplemental['Word'] = supplemental['Process'].copy()
    supplemental['Process'] = supplemental['Process_'].copy()
    supplemental.drop('Process_',axis=1,inplace=True)

    # Before Supplemental can be finalized, needs to merge into Final DF. to ORder BEFORE replacing Word with Word_
    consolidated_data = pd.concat([knowledge_base_df,supplemental])
    consolidated_data = consolidated_data.merge(processes[['PROC_ORDER',"Process"]].rename(columns={'Process':"Word"}),on='Word',how='left')

    cat_order_dict = {'Process':0,'Process Step':1}
    consolidated_data['CAT_ORDER'] = consolidated_data['Categorization'].map(cat_order_dict)
    # Fill as 0 so that Proceses will take higher priorirty, while retain NP Order
    consolidated_data['NP_ORDER'] = consolidated_data['NP_ORDER'].fillna(0)

    consolidated_data = consolidated_data.sort_values(['Process','PROC_ORDER','CAT_ORDER','NP_ORDER'])
    consolidated_data['Categorization'] = np.where(consolidated_data['Word_'].notnull(),consolidated_data['Word'],consolidated_data['Categorization'])
    consolidated_data['Word'] = np.where(consolidated_data['Word_'].notnull(),consolidated_data['Word_'],consolidated_data['Word'])

    return consolidated_data

    consolidated_data.drop(['Word_','PROC_ORDER','CAT_ORDER','NP_ORDER'],inplace=True,axis=1)

    
    # Create a DataFrame for the Highlevel Process, which is Processes and Process Steps.
    process_df = consolidated_data[consolidated_data['Categorization'].isin(['Process','Process Step'])].copy()

    if generate_excel_files:
        knowledge_base_df.to_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/knowledge_base.xlsx',index=False)
        process_df.to_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/defined_processes.xlsx',index=False)
        consolidated_data.to_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/consolidated_dataset.xlsx',index=False)
    
    return knowledge_base_df,process_df,consolidated_data

    

In [25]:
consolidated_df = generate_files_for_streamlit(definition_df,notes_df)
consolidated_df[consolidated_df['Process']=='Machine Learning Lifecycle']

,Process,Categorization,Word,Definition,NP_ORDER,Word_,PROC_ORDER,CAT_ORDER
46,Machine Learning Lifecycle,Process Step,Goal Setting,"Thought process utilized to establish a clear,...",0.0,NaN,0.0,1.0
65,Machine Learning Lifecycle,Goal Setting,Specific,Describes what will be accomplished and what s...,14.0,Specific,0.0,NaN
66,Machine Learning Lifecycle,Goal Setting,Measurable,Includes a clear way to track progress or dete...,15.0,Measurable,0.0,NaN
67,Machine Learning Lifecycle,Goal Setting,Achievable,"Realistic given available time, resources, ski...",16.0,Achievable,0.0,NaN
68,Machine Learning Lifecycle,Goal Setting,Relevant,"Aligns with broader priorities, such as team o...",17.0,Relevant,0.0,NaN
69,Machine Learning Lifecycle,Goal Setting,Time Bound,Includes a clear deadline or timeframe for com...,18.0,Time Bound,0.0,NaN
47,Machine Learning Lifecycle,Process Step,Problem Definition,"A Clear, Concise and explicit definition of th...",0.0,NaN,1.0,1.0
48,Machine Learning Lifecycle,Process Step,Data Collection,"Data collection involves identifying, sourcing...",0.0,NaN,2.0,1.0
49,Machine Learning Lifecycle,Process Step,Data Preparation,Data preparation focuses on cleaning and struc...,0.0,NaN,3.0,1.0
75,Machine Learning Lifecycle,Data Preparation,Regularization,,46.0,Regularization,3.0,1.0


In [6]:
knowledge_base_df,process_df,consolidated_df = generate_files_for_streamlit(definition_df,notes_df)


In [7]:
knowledge_base_df

,Process,Categorization,Word,Definition
0,Goal Setting,Process,Definition,"Thought process utilized to establish a clear,..."
1,Problem Definition,Process,Definition,"A Clear, Concise and explicit definition of th..."
2,Data Collection,Process,Definition,"Data collection involves identifying, sourcing..."
3,Data Preparation,Process,Definition,Data preparation focuses on cleaning and struc...
4,Exploratory Data Analysis,Process,Definition,Foundational step in ensuring the success of a...
...,...,...,...,...
60,Machine Learning Lifecycle,Process Step,Monitoring,Ongoing process of tracking model performance ...
61,Machine Learning Lifecycle,Process Step,"Bias, Fairness, and Ethics",This step examines whether the model produces ...
62,Model Selection,Process Step,Define Baseline,
63,Model Evaluation,Process Step,Test Relative to Baseline,


In [8]:
consolidated_df

,Process,Categorization,Word,Definition
18,Behavioural Economics,Definition,Anchoring Effect,A cognitive bias where people rely too heavily...
19,Behavioural Economics,Definition,Base Rate Fallacy,People ignore or undervalue general statistica...
20,Behavioural Economics,Definition,Bureaucratic Politics,Organization decision making paradigm in which...
21,Behavioural Economics,Definition,Biased Assimilation,Tendency to interpret information in a way tha...
22,Behavioural Economics,Definition,Blind spot bias,Cognitive bias where people are unaware of the...
...,...,...,...,...
44,Regularization,Data Engineering,Lasso,Technique used to prevent overfitting by addin...
45,Regularization,Data Engineering,Ridge,Technique used to prevent overfitting by addin...
8,Training,Process,Definition,Process of fitting a machine learning model to...
11,Validation,Process,Definition,Validation is the process of assessing model p...


In [9]:
process_df

,Process,Categorization,Word,Definition
15,"Bias, Fairness, and Ethics",Process,Definition,This step examines whether the model produces ...
2,Data Collection,Process,Definition,"Data collection involves identifying, sourcing..."
3,Data Preparation,Process,Definition,Data preparation focuses on cleaning and struc...
64,Data Preparation,Process Step,Regularization,
13,Deployment,Process,Definition,Deployment is the process of integrating the t...
4,Exploratory Data Analysis,Process,Definition,Foundational step in ensuring the success of a...
5,Feature Engineering,Process,Definition,Process of creating new input variables that b...
6,Feature Selection,Process,Definition,Process of identifying and retaining the most ...
0,Goal Setting,Process,Definition,"Thought process utilized to establish a clear,..."
9,Hyperparameter Tuning,Process,Definition,Optimizing the configuration settings that con...


In [4]:
definition_df.head(1)

,Process,Categorization,Word,Definition,Notes,Link,Image,Markdown Equation,Dataset Size,Learning Type,Algorithm Classification,Model Type
0,Behavioural Economics,Concept,Anchoring Effect,A cognitive bias where people rely too heavily...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
def missing_processes_from_knowledgebase(knowledge_base_df,definition_df):

    temp_df = definition_df[['Process','Categorization','Word',"Definition"]].copy()

    kb_temp = knowledge_base_df[['Process']].drop_duplicates().reset_index(drop=True)
    kb_temp['PROC_IN_KNOWLEDGE_BASE']= 1

    kb_temp1 = knowledge_base_df[['Word']].drop_duplicates().reset_index(drop=True)
    kb_temp1['WORD_IN_KNOWLEDGE_BASE']= 1

    temp_df = temp_df.merge(kb_temp,on='Process',how='left').merge(kb_temp1,on='Word',how='left').fillna(0)

    return temp_df

d = missing_processes_from_knowledgebase(knowledge_base_df,definition_df)

d[d['PROC_IN_KNOWLEDGE_BASE']==0]['Process'].value_counts()

Process
TBD                                        508
Mathematics                                 23
Generative Models                           22
General Definition                          17
Method Objective                            15
Representation Learning                     15
Information Retrieval                       14
Spark                                       12
Confusion Matrix                            12
Hyperparameter                              12
Kubernetes PE                               11
Dot Py String Classification                10
Wicked Problem                              10
Method Approach                             10
What Problem are we Trying to Solve?        10
Graph Algorithms                             9
Function                                     8
Time Series Forecasting                      8
Reinforcement Learning                       8
Optimization Function                        7
Classification                               5
Anoma

In [ ]:
auto_object_df = pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/object_auto_published.xlsx')
manual_object_df = pd.read_excel('/Users/derekdewald/Documents/Python/Github_Repo/Streamlit/Data/object_manual_published.xlsx')

In [ ]:
# I do not want to Bring Automated. It's confusing and causes Duplication. 
# I want to insure that Every Process from Definition Is Included.


# I should be able to Manage Everything from Notes.

# Process.
# Categorization - All Process Steps should be retained in Definition. Along with an Explanation as to what that process is.
# Word
# Definition. 

# Everything Should Have a Master Process. 

In [76]:
definition_df['Process'].value_counts()

Process
TBD                            508
Data Preparation                35
Mathematics                     23
Generative Models               22
General Definition              17
                              ... 
Decison Making                   1
Natural Language Processing      1
Langchain                        1
Nonlinear Models                 1
Exploratory Data Analysis        1
Name: count, Length: 65, dtype: int64

In [ ]:
# Process >> Process Step >> __________

In [160]:
generate_files_for_streamlit(definition_df,notes_df)

ValueError: operands could not be broadcast together with shapes (76,) (44,) (76,) 

In [150]:
notes_df = pd.read_csv(object_dict['csv_links']['python_object']['google_notes_csv'])
definition_df = pd.read_csv(object_dict['csv_links']['python_object']['google_definition_csv'])


In [156]:
#knowledge_base_df,processes,not_processes = generate_files_for_streamlit(definition_df,notes_df)

ValueError: cannot reindex on an axis with duplicate labels

In [140]:
knowledge_base_df.iloc[:30]

,Process,Categorization,Word,Definition
0,Goal Setting,Process,Definition,"Thought process utilized to establish a clear,..."
1,Problem Definition,Process,Definition,"A Clear, Concise and explicit definition of th..."
2,Data Collection,Process,Definition,"Data collection involves identifying, sourcing..."
3,Data Preparation,Process,Definition,Data preparation focuses on cleaning and struc...
4,Exploratory Data Analysis,Process,Definition,Foundational step in ensuring the success of a...
5,Feature Engineering,Process,Definition,Process of creating new input variables that b...
6,Feature Selection,Process,Definition,Process of identifying and retaining the most ...
7,Validation/ Model Selection,Process,Definition,The process of choosing the most appropriate a...
8,Training,Process,Definition,Process of fitting a machine learning model to...
9,Hyperparameter Tuning,Process,Definition,Optimizing the configuration settings that con...


In [141]:
knowledge_base_df.iloc[30:]

,Process,Categorization,Word,Definition
30,Behavioural Economics,Definition,Naive realism,Naïve realism is a concept from social psychol...
31,Behavioural Economics,Definition,Ostrich Effect,cognitive bias where people avoid information ...
32,Goal Setting,Requirement,Specific,Describes what will be accomplished and what s...
33,Goal Setting,Requirement,Measurable,Includes a clear way to track progress or dete...
34,Goal Setting,Requirement,Achievable,"Realistic given available time, resources, ski..."
35,Goal Setting,Requirement,Relevant,"Aligns with broader priorities, such as team o..."
36,Goal Setting,Requirement,Time Bound,Includes a clear deadline or timeframe for com...
37,General Definition,Definition,Baseline,Baseline represents the typical or expected be...
38,General Definition,Definition,Dimension,A Dimension is a conceptual axis of behavior o...
39,General Definition,Definition,Implementation,An Implementation is a specific measurable rep...


In [136]:
process_df

,Process,Categorization,Word,Definition
0,"Bias, Fairness, and Ethics",Process,Definition,This step examines whether the model produces ...
1,Data Collection,Process,Definition,"Data collection involves identifying, sourcing..."
2,Data Preparation,Process,Definition,Data preparation focuses on cleaning and struc...
3,Deployment,Process,Definition,Deployment is the process of integrating the t...
4,Exploratory Data Analysis,Process,Definition,Foundational step in ensuring the success of a...
5,Feature Engineering,Process,Definition,Process of creating new input variables that b...
6,Feature Selection,Process,Definition,Process of identifying and retaining the most ...
7,Goal Setting,Process,Definition,"Thought process utilized to establish a clear,..."
13,Hyperparameter Tuning,Process,Definition,Optimizing the configuration settings that con...
15,Machine Learning Lifecycle,Process Step,Goal Setting,"Thought process utilized to establish a clear,..."


In [138]:
consolidated_df

,Process,Categorization,Word,Definition
0,"Bias, Fairness, and Ethics",Process,Definition,This step examines whether the model produces ...
1,Data Collection,Process,Definition,"Data collection involves identifying, sourcing..."
2,Data Preparation,Process,Definition,Data preparation focuses on cleaning and struc...
3,Deployment,Process,Definition,Deployment is the process of integrating the t...
4,Exploratory Data Analysis,Process,Definition,Foundational step in ensuring the success of a...
5,Feature Engineering,Process,Definition,Process of creating new input variables that b...
6,Feature Selection,Process,Definition,Process of identifying and retaining the most ...
7,Goal Setting,Process,Definition,"Thought process utilized to establish a clear,..."
8,Goal Setting,Requirement,Specific,Describes what will be accomplished and what s...
9,Goal Setting,Requirement,Measurable,Includes a clear way to track progress or dete...
